# CAMEL Ambient Solar Wind Validation

## Overview

This notebook demonstrates how to validate ambient solar wind models against in situ measurements near Earth. The workflow is based on the **CAMEL (Comprehensive Assessment of Models and Events using Library tools)** framework developed at the [Community Coordinated Modeling Center (CCMC)](https://ccmc.gsfc.nasa.gov/).

This effort is part of the international [ISWAT H1-01 Team](https://iswat-cospar.org/h1-01), which aims to establish a unified validation approach for solar wind models across the heliophysics and space weather communities.

The validation procedures presented in this notebook series are also available through an interactive web application built with Plotly Dash and Python. The **Ambient Solar Wind Validation Dashboard** can be accessed here:

- [CCMC CAMEL Ambient Solar Wind Validation Dashboard](https://ccmc.gsfc.nasa.gov/camel/AmbientSolarWind/home)

These notebooks provide a reproducible version of the same validation workflow and are intended to help users explore, reproduce, and extend the validation procedures of the CCMC CAMEL dashboard.

**Notebook version:** 1.0

---

## Notebook Scope

### 04 Peak Analysis

Peak analysis evaluates how well models reproduce **significant solar wind speed enhancements**, focusing on event timing and amplitude rather than every individual time step.

This notebook:
- loads the preprocessed example datasets generated in `01_api_and_preprocessing.ipynb`
- identifies significant solar wind speed peaks in observed and modeled time series
- evaluates model performance using event-based peak metrics
- reproduces the peak-analysis visualization workflow used in the dashboard

---

## Expected Inputs

This notebook expects the following processed datasets:

```text
../example_data/processed/
├── omni_obs.csv
├── ambsowi_mod.csv
├── bravda_mod.csv
└── p27_mod.csv
```

## Imports

We import the metric and visual diagnostic functions from `utils/`. Note that these functions are an exact copy of the functions that are used in the CAMEL Ambient Solar Wind Dashboard. 

In [ ]:
import sys
from pathlib import Path

project_root = ".."
utils_path = "../utils"

if project_root not in sys.path:
    sys.path.append(project_root)
if utils_path not in sys.path:
    sys.path.append(utils_path)

import numpy as np
import pandas as pd
from IPython.display import display

from skill_scores import *
from utils.visualizations import *


## Load the processed example datasets

These CSV files are assumed to have already been preprocessed and aligned in `01_api_and_preprocessing.ipynb`. Note that if you want to change the selected time window for validation you need to go back to `01_api_and_preprocessing.ipynb`.


In [ ]:
data_dir = Path("../example_data/processed")

omni_obs = pd.read_csv(data_dir / "omni_obs.csv", parse_dates=["Time"], index_col="Time")
ambsowi_mod = pd.read_csv(data_dir / "ambsowi_mod.csv", parse_dates=["Time"], index_col="Time")
bravda_mod = pd.read_csv(data_dir / "bravda_mod.csv", parse_dates=["Time"], index_col="Time")
p27_mod = pd.read_csv(data_dir / "p27_mod.csv", parse_dates=["Time"], index_col="Time")

print("Observed shape:   ", omni_obs.shape)
print("AmbSoWi-ML shape: ", ambsowi_mod.shape)
print("BRaVDA shape:     ", bravda_mod.shape)
print("Persistence shape:", p27_mod.shape)

## Prepare inputs for the peak-analysis workflow

Convert the processed time series into the long-format structure expected by the CAMEL dashboard functions.

In [ ]:
def to_pred(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Model"})
    d["Name"] = name
    return d[["Time", "Model", "Name"]]

def to_obs(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Obs"})
    d["Name"] = name
    return d[["Time", "Obs", "Name"]]

observed_data = to_obs(omni_obs, "OMNI2 | Hourly averaged")

predicted_ambsowi = to_pred(ambsowi_mod, "AmbSoWi-ML")
predicted_bravda = to_pred(bravda_mod, "BRaVDA")
baseline_data = to_pred(p27_mod, "Persistence (27-days)")

## Compute event-based peak metrics

Peak analysis focuses on the model’s ability to **capture solar wind speed peaks**. First, we compute matched hits, misses, and false alarms from detected peaks, then derive several event-based metrics.

To get started, we need to define an **event threshold** as well as a **time window** for the peak search algorithm. 



In [ ]:
threshold_value = 450   # km/s minimum speed for peak detection
maximum_dt = 2          # days maximum time difference allowed for matching peaks

print("Peak threshold:", threshold_value)
print("Maximum matching time difference [days]:", maximum_dt)

In [ ]:
def compute_peak_metrics(
    predicted: pd.DataFrame,
    observed: pd.DataFrame,
    threshold_value: float,
    maximum_dt: float,
):
    _, _, hits, misses, false_alarms, _, _ = compute_peaks(
        predicted["Time"],
        observed["Obs"],
        predicted["Model"],
        threshold_value,
        maximum_dt,
    )

    contingency_table = pd.DataFrame(
        [[len(hits), len(false_alarms), len(misses), 0]],
        columns=["tp", "fp", "fn", "tn"],
    )

    metrics = {
        "True Positive Rate": truePositiveRate(contingency_table)[0],
        "False Negative Rate": falseNegativeRate(contingency_table)[0],
        "Positive Predictive Value": positivePredictiveValue(contingency_table)[0],
        "False Alarm Ratio": falseAlarmRatio(contingency_table)[0],
        "Threat Score": threatScore(contingency_table)[0],
        "Bias": bias(contingency_table)[0],
    }

    metrics_table = pd.DataFrame.from_dict(
        metrics,
        orient="index",
        columns=["Value"],
    )

    return contingency_table, metrics_table


ct_ambsowi, peak_metrics_ambsowi = compute_peak_metrics(
    predicted_ambsowi,
    observed_data,
    threshold_value,
    maximum_dt,
)

ct_persistence, peak_metrics_persistence = compute_peak_metrics(
    baseline_data,
    observed_data,
    threshold_value,
    maximum_dt,
)

print("AmbSoWi-ML contingency table")
display(ct_ambsowi)

print("AmbSoWi-ML peak metrics")
display(peak_metrics_ambsowi.round(2))

print("Persistence baseline contingency table")
display(ct_persistence)

print("Persistence baseline peak metrics")
display(peak_metrics_persistence.round(2))

## Compare event-based peak metrics across all available models

This table extends the original single-model example to all available model series in `example_data/`.

In [ ]:
def compute_peak_metric_table(models: dict, observed_long: pd.DataFrame, threshold_value: float, maximum_dt: float) -> pd.DataFrame:
    rows = {}
    for name, pred_long in models.items():
        _, _, hits, misses, false_alarms, _, _ = compute_peaks(
            pred_long["Time"],
            observed_long["Obs"],
            pred_long["Model"],
            threshold_value,
            maximum_dt
        )
        ct = pd.DataFrame([[len(hits), len(false_alarms), len(misses), 0]],
                          columns=["tp", "fp", "fn", "tn"])
        rows[name] = {
            "tp": int(ct["tp"].iloc[0]),
            "fp": int(ct["fp"].iloc[0]),
            "fn": int(ct["fn"].iloc[0]),
            "TPR": float(truePositiveRate(ct)[0]),
            "FNR": float(falseNegativeRate(ct)[0]),
            "PPV": float(positivePredictiveValue(ct)[0]),
            "FAR": float(falseAlarmRatio(ct)[0]),
            "TS": float(threatScore(ct)[0]),
            "Bias": float(bias(ct)[0]),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

peak_metric_table = compute_peak_metric_table(
    {
        "AmbSoWi-ML": predicted_ambsowi,
        "BRaVDA": predicted_bravda,
        "Persistence (27d)": baseline_data,
    },
    observed_data,
    threshold_value,
    maximum_dt
)

display(peak_metric_table.round(2))


## Visual Diagnostics for Peak Analysis

The visualizations in this section include:
- timelines with peak markers
- bar charts summarizing hits, misses, and false alarms
- histograms of timing and magnitude errors
- scatter plots for event relationships and error diagnostics

In [ ]:
fig = create_timeline_with_peaks(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    observed_data["Name"][0],
    predicted_ambsowi["Name"][0],
    threshold_value,
    maximum_dt
)

fig.update_layout(width=1200, height=520)
fig.show()

In [ ]:
fig = create_peak_bar_chart(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt
)

fig.update_layout(width=400, height=400)
fig.show()


In [ ]:
fig = create_histogram_plot(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt,
    x_label="Error [km s⁻¹]",
    plot_type="Data Diff"
)

fig.update_layout(width=500, height=500)
fig.show()


In [ ]:
fig = create_histogram_plot(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt,
    x_label="Time Error [hours]",
    plot_type="Time Error [hours]"
)

fig.update_layout(width=500, height=500)
fig.show()


In [ ]:
fig = create_scatter_plot_for_hits(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt,
    max_points=1000,
    min_points=50,
    opacity=0.5,
    x_label="Observed Bulk Speed [km s⁻¹]",
    y_label="Predicted Bulk Speed [km s⁻¹]",
    plot_title="Model Hits: Observed vs Predicted"
)

fig.update_layout(width=500, height=500)
fig.show()


In [ ]:
fig = create_error_scatter_plot(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt,
    x_label="Time Error [hours]",
    y_label="Speed Error [km s⁻¹]",
    plot_title="Model Event Timing vs Magnitude Errors",
    max_points=1000,
    min_points=50,
    opacity=0.5,
    time_bounds=(-96, 96),
    data_bounds=(-400, 400)
)

fig.update_layout(width=500, height=500)
fig.show()


In [ ]:
fig = create_time_error_vs_observed_data_plot(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt,
    x_label="Time Error [hours]",
    y_label="Observed Bulk Speed [km s⁻¹]",
    plot_title="Time Error vs. Observed Bulk Speed",
    max_points=1000,
    min_points=50,
    opacity=0.5,
    time_bounds=(-96, 96),
    obs_bounds=(200, 800)
)

fig.update_layout(width=500, height=500)
fig.show()


## PDF export example

If `kaleido` is installed, Plotly figures can be exported as vector PDFs.

In [ ]:
import os

out_dir = "../results/output_pdfs"
os.makedirs(out_dir, exist_ok=True)

fig = create_peak_bar_chart(
    predicted_ambsowi["Time"],
    observed_data["Obs"],
    predicted_ambsowi["Model"],
    threshold_value,
    maximum_dt
)
fig.update_layout(width=400, height=400)

pdf_path = os.path.join(out_dir, "peak_bar_chart_ambsowi.pdf")

# Requires kaleido:
fig.write_image(pdf_path)

print("Saved:", pdf_path)

Save all visualizations generated in this notebook as PDF files in `../results/output_pdfs/`. This requires `kaleido`.

In [ ]:
from pathlib import Path

out_dir = Path("../results/output_pdfs")
out_dir.mkdir(parents=True, exist_ok=True)

figures = {
    "timeline_with_peaks_ambsowi": create_timeline_with_peaks(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        observed_data["Name"][0],
        predicted_ambsowi["Name"][0],
        threshold_value,
        maximum_dt,
    ),
    "peak_bar_chart_ambsowi": create_peak_bar_chart(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        threshold_value,
        maximum_dt,
    ),
    "peak_data_error_histogram_ambsowi": create_histogram_plot(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        threshold_value,
        maximum_dt,
        x_label="Error [km s⁻¹]",
        plot_type="Data Diff",
    ),
    "peak_time_error_histogram_ambsowi": create_histogram_plot(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        threshold_value,
        maximum_dt,
        x_label="Time Error [hours]",
        plot_type="Time Error [hours]",
    ),
    "peak_hits_scatter_ambsowi": create_scatter_plot_for_hits(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        threshold_value,
        maximum_dt,
        max_points=1000,
        min_points=50,
        opacity=0.5,
        x_label="Observed Bulk Speed [km s⁻¹]",
        y_label="Predicted Bulk Speed [km s⁻¹]",
        plot_title="Model Hits: Observed vs Predicted",
    ),
    "peak_error_scatter_ambsowi": create_error_scatter_plot(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        threshold_value,
        maximum_dt,
        x_label="Time Error [hours]",
        y_label="Speed Error [km s⁻¹]",
        plot_title="Model Event Timing vs Magnitude Errors",
        max_points=1000,
        min_points=50,
        opacity=0.5,
        time_bounds=(-96, 96),
        data_bounds=(-400, 400),
    ),
    "peak_time_error_vs_observed_ambsowi": create_time_error_vs_observed_data_plot(
        predicted_ambsowi["Time"],
        observed_data["Obs"],
        predicted_ambsowi["Model"],
        threshold_value,
        maximum_dt,
        x_label="Time Error [hours]",
        y_label="Observed Bulk Speed [km s⁻¹]",
        plot_title="Time Error vs. Observed Bulk Speed",
        max_points=1000,
        min_points=50,
        opacity=0.5,
        time_bounds=(-96, 96),
        obs_bounds=(200, 800),
    ),
}

figure_sizes = {
    "timeline_with_peaks_ambsowi": (1200, 520),
    "peak_bar_chart_ambsowi": (400, 400),
    "peak_data_error_histogram_ambsowi": (500, 500),
    "peak_time_error_histogram_ambsowi": (500, 500),
    "peak_hits_scatter_ambsowi": (500, 500),
    "peak_error_scatter_ambsowi": (500, 500),
    "peak_time_error_vs_observed_ambsowi": (500, 500),
}

for name, fig in figures.items():
    width, height = figure_sizes[name]
    fig.update_layout(width=width, height=height)

    pdf_path = out_dir / f"{name}.pdf"

    # Requires kaleido:
    fig.write_image(pdf_path)

    print("Saved:", pdf_path)